# Что видит классификатор

Показывает по каждому кадру: что сеть ответила, насколько уверена и **куда смотрела**.

Карта внимания здесь не приблизительная. Голова у модели линейная и стоит поверх
признаков, усреднённых по кадру, — а для такой схемы вклад каждого участка считается
теми же весами головы, только без усреднения. То есть подсвечены буквально те места,
из-за которых модель поставила свою оценку.

Перед запуском нужна обученная модель:

```bash
uv run adlearn cls features
uv run adlearn cls train
```


## Настройки


In [ ]:
from pathlib import Path

# Папка с фотографиями, которые хотим разобрать.
SOURCE = Path("notebooks")

MODEL = Path("data/classification/model.joblib")
LIMIT = 20  # сколько кадров показать
SEED = 0  # перемешивание, чтобы не смотреть всегда одни и те же
MIN_CONFIDENCE = 0.55  # ниже — ответ «не уверен»
DEVICE = "cuda"  # 'cpu', если GPU занята

ONLY_MISTAKES = False  # True — показать только ошибки

# Правильные ответы, если папка не названа именем класса.
# Ключ — имя файла без расширения. Что не перечислено — считается неизвестным.
TRUTH = {
    "image-1": "beeline",
    "image-2": "beeline",
    "image-3": "tele2",
    "image-4": "tele2",
    "image-5": "megafon",
    "image-6": "megafon",
    "image-7": "megafon",
    "image": "tele2",
}

## Загрузка


In [ ]:
import random

import joblib
import matplotlib.pyplot as plt

from adlearn.classification.config import UNSURE
from adlearn.classification.explain import Backbone, explain, overlay
from adlearn.core.images import find_images

bundle = joblib.load(MODEL)
BRANDS = tuple(bundle["brands"])

paths = find_images(SOURCE, recursive=True)
random.Random(SEED).shuffle(paths)
paths = paths[:LIMIT]

backbone = Backbone(device=DEVICE)
results = explain(paths, bundle, device=DEVICE, backbone=backbone)


def truth_of(path):
    """Правильный ответ: из TRUTH, иначе из имени папки, иначе неизвестен."""
    if path.stem in TRUTH:
        return TRUTH[path.stem]
    return path.parent.name if path.parent.name in BRANDS else None


known = sum(truth_of(p) is not None for p in paths)
print(f"кадров: {len(results)}   классы: {BRANDS}")
print(f"с известным ответом: {known} из {len(paths)}")

## Разбор по кадрам

Слева — кадр так, как его увидела сеть (вписан в квадрат с белыми полями).
В центре — карта внимания для победившего класса: красное значит «вот из-за этого».
Справа — вероятности всех классов и то, что намерил цветовой модуль.


In [ ]:
COLOR_ROWS = ("beeline_score", "megafon_score", "tele2_score")


def show(item):
    brand, confidence = item.answer
    answer = brand if confidence >= MIN_CONFIDENCE else UNSURE
    truth = truth_of(item.path)
    correct = None if truth is None else (brand == truth)

    figure, axes = plt.subplots(1, 3, figsize=(13, 4.2), gridspec_kw={"width_ratios": [1, 1, 1.15]})

    axes[0].imshow(item.frame)
    head = item.path.name[:30]
    axes[0].set_title(head if truth is None else f"{head}  (правда: {truth})", fontsize=9)

    axes[1].imshow(overlay(item.frame, item.heatmaps[brand]))
    axes[1].set_title(f"внимание: {brand}", fontsize=9)

    for axis in axes[:2]:
        axis.set_xticks([])
        axis.set_yticks([])

    shade = "#c0392b" if correct is False else ("#27ae60" if correct else "#2c7fb8")
    axes[2].barh(list(BRANDS)[::-1], item.probabilities[::-1], color=shade)
    axes[2].set_xlim(0, 1)
    axes[2].axvline(MIN_CONFIDENCE, color="#888", linestyle="--", linewidth=1)
    mark = "" if correct is None else ("  ✓" if correct else "  ✗")
    axes[2].set_title(f"{answer}  {confidence:.2f}{mark}", fontsize=10)
    axes[2].tick_params(labelsize=9)

    palette = "   ".join(f"{k.split('_')[0]} {item.color[k]:.2f}" for k in COLOR_ROWS)
    figure.text(0.665, -0.02, "цветовой модуль:  " + palette, fontsize=8.5, color="#555")

    plt.tight_layout()
    plt.show()


shown = 0
for item in results:
    if ONLY_MISTAKES and item.answer[0] == truth_of(item.path):
        continue
    show(item)
    shown += 1
print(f"показано {shown} из {len(results)}")

## Внимание по всем классам

Иногда полезнее увидеть, где модель искала каждый бренд — особенно на ошибках.
Меняй номер кадра и смотри, что перетянуло ответ.


In [ ]:
INDEX = 0

item = results[INDEX]
figure, axes = plt.subplots(1, len(BRANDS) + 1, figsize=(3.1 * (len(BRANDS) + 1), 3.4))
axes[0].imshow(item.frame)
axes[0].set_title(item.path.name[:26], fontsize=9)
for axis, brand in zip(axes[1:], BRANDS, strict=True):
    axis.imshow(overlay(item.frame, item.heatmaps[brand]))
    axis.set_title(f"{brand}  {item.probabilities[BRANDS.index(brand)]:.2f}", fontsize=9)
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.tight_layout()
plt.show()

## Что намерил цветовой модуль

Все 55 цветовых признаков одного кадра. Полезно, когда непонятно, почему модель
потянуло не туда: видно, какие доли и палитры сработали.


In [ ]:
item = results[INDEX]
interesting = {
    k: v
    for k, v in item.color.items()
    if k.endswith(("_score", "_balance", "_coverage_peak")) or k.startswith("peak_")
}
names = list(interesting)[::-1]
values = [interesting[n] for n in names]

plt.figure(figsize=(7, 0.24 * len(names) + 1))
plt.barh(names, values, color="#2c7fb8")
plt.xlim(0, 1)
plt.title(f"цветовые признаки: {item.path.name[:34]}", fontsize=10)
plt.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

## Сводка по папке

Если папка названа именем класса — считает, сколько угадано и куда уехало остальное.


In [ ]:
from collections import Counter

answers = Counter()
hits = graded = 0
mistakes = []
for item in results:
    brand, confidence = item.answer
    answers[brand if confidence >= MIN_CONFIDENCE else UNSURE] += 1
    truth = truth_of(item.path)
    if truth is not None:
        graded += 1
        if brand == truth:
            hits += 1
        else:
            mistakes.append((item.path.name, truth, brand, confidence))

print("ответы модели:")
for name, count in answers.most_common():
    print(f"  {name:12} {count:4}  {count / len(results):5.0%}")

if graded:
    print(f"\nверно {hits}/{graded} = {hits / graded:.0%}")
    if mistakes:
        print("\nошибки:")
        for name, truth, brand, confidence in mistakes:
            print(f"  {name:24} правда {truth:9} ответ {brand:9} {confidence:.2f}")
else:
    print("\nправильные ответы не заданы — заполни TRUTH, чтобы увидеть точность.")